In [9]:
from glob import glob
import os
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
input_dir = "data/dataset/train"
output_dir = "data/dataset_crops_sat"
crop_size = 32

# images annotées de concepts
# image_paths = [
#     p for p in glob(os.path.join(input_dir, "**", "*Esri10*"), recursive=True)
#     if p.lower().endswith((".jpg", ".jpeg", ".png"))
# ]

# images satellites brutes
image_paths = [
    p for p in glob(os.path.join(input_dir, "**", "*S2L2A*"), recursive=True)
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
]

print(len(image_paths), "images found.")

os.makedirs(output_dir, exist_ok=True)

def get_label(pixel):
    if pixel == (0, 0, 0):
        return "0"
    elif pixel == (181, 254, 142):
        return "1"
    elif pixel == (254, 217, 142):
        return "2"
    elif pixel == (254, 153, 41):
        return "3"
    elif pixel == (204, 76, 2):
        return "4"
    elif pixel == (255, 255, 255):
        return "0"
    else:
        return print("erreur, unknown label:", pixel)

for path_to_img in tqdm(image_paths):
    img = Image.open(path_to_img).convert("RGB")

    path_to_img_gra = path_to_img.replace("S2L2A", "GRA")
    path_to_img_del = path_to_img.replace("S2L2A", "DEL")
    if os.path.exists(path_to_img_gra):
        path_to_img_burned = path_to_img_gra
    elif os.path.exists(path_to_img_del):
        path_to_img_burned = path_to_img_del
    else:
        print("********** image brûlée non trouvée *********")
        continue
    img_burn = Image.open(path_to_img_burned).convert("RGB")

    width, height = img.size

    # Calculer combien de crops on peut faire horizontalement et verticalement
    n_crops_x = width // crop_size
    n_crops_y = height // crop_size

    base_name = os.path.splitext(os.path.basename(path_to_img))[0]

    # Découper en crops 32x32
    for i in range(n_crops_y):
        for j in range(n_crops_x):
            left = j * crop_size
            upper = i * crop_size
            right = left + crop_size
            lower = upper + crop_size

            crop = img.crop((left, upper, right, lower))

            crop_burn = img_burn.crop((left, upper, right, lower))
            middle_pixel = crop_burn.getpixel((crop_size // 2, crop_size // 2))
            label = get_label(middle_pixel)

            # Création du dossier de la classe si besoin
            label_dir = os.path.join(output_dir, label)
            os.makedirs(label_dir, exist_ok=True)


            # Nom unique pour le crop
            crop_filename = f"{base_name}_crop_{i}_{j}.png"
            crop.save(os.path.join(label_dir, crop_filename))


281 images found.


100%|██████████| 281/281 [01:30<00:00,  3.10it/s]
